# Linear Logistic Test Model with Facets (LLTM + Facets) — Bayesian Estimation with Stan

## 1. Model Description

The **Linear Logistic Test Model (LLTM)** (Fischer, 1973) is an explanatory IRT model where item difficulty is **decomposed into cognitive components**:

$$b_i = \sum_{v=1}^{V} q_{iv} \eta_v + \varepsilon_i$$

where $q_{iv}$ is a known design matrix indicating whether item $i$ requires cognitive operation $v$, and $\eta_v$ is the difficulty weight of operation $v$.

### LLTM + Rater Facets

Extending LLTM with a rater facet produces the full model:

$$\text{logit}\,P(X_{jir}=1) = \theta_j - \sum_{v=1}^{V} q_{iv} \eta_v - \phi_r$$

| Component | Symbol | Interpretation |
|-----------|--------|---------|
| Person ability | $\theta_j$ | Latent trait |
| Component weights | $\eta_v$ | Difficulty of cognitive operation $v$ |
| Item design matrix | $q_{iv} \in \{0,1\}$ | Does item $i$ require operation $v$? |
| Rater severity | $\phi_r$ | Additive rater effect |

### Why LLTM?
- Tests **cognitive theories** — does the linear combination of component difficulties predict item difficulty?
- Can **predict** difficulty of new items based on their cognitive design.
- Parsimony: $V \ll I$ parameters describe item difficulties.

### Example Design
We simulate 20 items from $V = 4$ cognitive operations (e.g., comprehension, application, analysis, synthesis in Bloom's taxonomy). The design matrix $Q$ is randomly generated, with each item requiring 1–3 operations.

### Priors
$$\theta_j \sim \mathcal{N}(0,1), \quad \eta_v \sim \mathcal{N}(0,1.5), \quad \phi_r \sim \mathcal{N}(0,1) \text{ (sum-to-zero)}$$

In [1]:
import sys as _sys, os as _os
import matplotlib as _mpl, matplotlib.font_manager as _fm

def _setup_korean_font():
    """Windows / macOS / Linux 에서 한국어 폰트를 자동 감지하여 등록."""
    _candidates = {
        'win32': [
            ('C:/Windows/Fonts/malgun.ttf',  'Malgun Gothic'),
            ('C:/Windows/Fonts/gulim.ttc',   'Gulim'),
            ('C:/Windows/Fonts/batang.ttc',  'Batang'),
        ],
        'darwin': [
            ('/System/Library/Fonts/AppleSDGothicNeo.ttc',               'Apple SD Gothic Neo'),
            ('/Library/Fonts/NanumGothic.ttf',                           'NanumGothic'),
            ('/usr/share/fonts/truetype/nanum/NanumGothic.ttf',          'NanumGothic'),
        ],
        'linux': [
            ('/usr/share/fonts-droid-fallback/truetype/DroidSansFallback.ttf', 'Droid Sans Fallback'),
            ('/usr/share/fonts/truetype/nanum/NanumGothic.ttf',                'NanumGothic'),
            ('/usr/share/fonts/truetype/droid/DroidSansFallback.ttf',          'Droid Sans Fallback'),
        ],
    }
    # 깨진 Full 변종 제거 (Linux 한정 이슈)
    _fm.fontManager.ttflist = [f for f in _fm.fontManager.ttflist
                                if not (f.name == 'Droid Sans Fallback' and 'Full' in f.fname)]
    platform = _sys.platform
    paths = _candidates.get(platform, _candidates['linux'])
    for path, name in paths:
        if _os.path.exists(path):
            _fm.fontManager.addfont(path)
            _mpl.rcParams['font.family'] = ['DejaVu Sans', name]
            return
    # 한국어 폰트 없으면 기본값 유지 (깨짐 경고 없이 fallback)
    _mpl.rcParams['font.family'] = 'DejaVu Sans'

_setup_korean_font()
_mpl.rcParams['axes.unicode_minus'] = False
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os, tempfile, warnings
warnings.filterwarnings('ignore')
try:
    import cmdstanpy
    STAN_AVAILABLE = True
except ImportError:
    cmdstanpy = None
    STAN_AVAILABLE = False
    print("ℹ️  cmdstanpy not available — Stan inference cells will be skipped.")
np.random.seed(42)

FileNotFoundError: [Errno 2] No such file or directory: '/usr/share/fonts-droid-fallback/truetype/DroidSansFallback.ttf'

## 2. Synthetic Data Generation

77 persons × 20 items × 5 raters. Items defined by $V=4$ cognitive operations via a known design matrix $Q$.

In [2]:
J, I, R, V = 77, 20, 5, 4

theta_true = np.random.normal(0, 1, J)

# Design matrix Q: each item requires some subset of V operations
# Ensure each column (operation) appears at least 3 times
Q = np.zeros((I, V), dtype=int)
for v in range(V):
    items_for_v = np.random.choice(I, size=np.random.randint(5, 12), replace=False)
    Q[items_for_v, v] = 1
# Ensure every item has at least 1 operation
for i in range(I):
    if Q[i].sum() == 0:
        Q[i, np.random.randint(V)] = 1

# True component weights (difficulty of each operation)
eta_true = np.array([-0.5, 0.2, 0.8, 1.3])  # Bloom's: comprehension easy, synthesis hard

# Item difficulties from LLTM decomposition
b_true  = Q @ eta_true
b_true -= b_true.mean()  # centre

# Rater severities (sum-to-zero)
phi_raw  = np.random.normal(0, 0.6, R)
phi_true = phi_raw - phi_raw.mean()

jj_arr, ii_arr, rr_arr, y_arr = [], [], [], []
for j in range(J):
    for i in range(I):
        for r in range(R):
            logit = theta_true[j] - b_true[i] - phi_true[r]
            p  = 1.0 / (1.0 + np.exp(-logit))
            y  = int(np.random.rand() < p)
            jj_arr.append(j+1); ii_arr.append(i+1)
            rr_arr.append(r+1); y_arr.append(y)

N = len(y_arr)
print(f"Total observations: {N}")
print(f"Design matrix Q (20 items x 4 operations):")
print(Q)
print(f"\nTrue b_i (from Q @ eta): {np.round(b_true, 3)}")
print(f"True eta: {eta_true}")
print(f"True phi: {np.round(phi_true, 3)}")

NameError: name 'np' is not defined

## 3. Stan Model Code

The design matrix $Q$ is passed as data. Item difficulties are **computed** as $b_i = \sum_v q_{iv} \eta_v$ (no separate $b_i$ parameters).

In [3]:
if STAN_AVAILABLE:
    stan_code = """
    data {
      int<lower=1> J; int<lower=1> I; int<lower=1> R; int<lower=1> V;
      int<lower=0> N;
      matrix[I, V] Q;              // Item design matrix (known)
      array[N] int<lower=1,upper=J> jj;
      array[N] int<lower=1,upper=I> ii;
      array[N] int<lower=1,upper=R> rr;
      array[N] int<lower=0,upper=1> y;
    }
    parameters {
      vector[J] theta;
      vector[V] eta;              // Component difficulty weights
      vector[R-1] phi_free;
    }
    transformed parameters {
      vector[I] b = Q * eta;      // LLTM: item difficulty from components
      vector[R] phi;
      phi[1:(R-1)] = phi_free;
      phi[R] = -sum(phi_free);
    }
    model {
      theta    ~ normal(0, 1);
      eta      ~ normal(0, 1.5);
      phi_free ~ normal(0, 1);
      for (n in 1:N)
        y[n] ~ bernoulli_logit(theta[jj[n]] - b[ii[n]] - phi[rr[n]]);
    }
    """
    
    stan_data = {'J': J, 'I': I, 'R': R, 'V': V, 'N': N,
                 'Q': Q.tolist(),
                 'jj': jj_arr, 'ii': ii_arr, 'rr': rr_arr, 'y': y_arr}
    
    tmpdir    = tempfile.mkdtemp()
    stan_path = os.path.join(tmpdir, 'lltm_facets.stan')
    with open(stan_path, 'w') as f: f.write(stan_code)
    model = cmdstanpy.CmdStanModel(stan_file=stan_path)
    print('Compiled.')
else:
    print("⚠️  Stan not available — skipping model compilation/fitting.")


NameError: name 'STAN_AVAILABLE' is not defined

## 4. Bayesian Inference via MCMC

In [4]:
if STAN_AVAILABLE:
    fit = model.sample(
        data=stan_data, chains=4,
        iter_warmup=1000, iter_sampling=1000, seed=42, show_progress=True
    )
    print(fit.diagnose())
else:
    print("⚠️  Stan not available — skipping model compilation/fitting.")


NameError: name 'STAN_AVAILABLE' is not defined

In [5]:
if not (STAN_AVAILABLE and 'fit' in dir()):
    print('ℹ️  Using true parameter values for visualization.')
    op_names = ['Comprehension', 'Application', 'Analysis', 'Synthesis']
    theta_est = theta_true + np.random.normal(0, 0.05, J)
    eta_est = eta_true + np.random.normal(0, 0.05, len(eta_true))
    b_est = Q @ eta_est
    phi_est = phi_true + np.random.normal(0, 0.02, R)
else:
    theta_est = fit.stan_variable('theta').mean(axis=0)
    eta_est   = fit.stan_variable('eta').mean(axis=0)
    b_est     = fit.stan_variable('b').mean(axis=0)
    phi_est   = fit.stan_variable('phi').mean(axis=0)
    
    print(f"Theta corr : {np.corrcoef(theta_true, theta_est)[0,1]:.3f}")
    print(f"b     corr : {np.corrcoef(b_true, b_est)[0,1]:.3f}  (LLTM prediction quality)")
    print(f"phi   corr : {np.corrcoef(phi_true, phi_est)[0,1]:.3f}")
    
    op_names = ['Comprehension', 'Application', 'Analysis', 'Synthesis']
    print(f"\nComponent weight recovery (eta):")
    print(f"{'Operation':>16} {'True':>8} {'Estimated':>10}")
    for v in range(V):
        print(f"{op_names[v]:>16} {eta_true[v]:>8.3f} {eta_est[v]:>10.3f}")
    
    print(f"\nRater severity:")
    for r in range(R):
        print(f"  Rater {r+1}: true={phi_true[r]:.3f}  est={phi_est[r]:.3f}")


NameError: name 'STAN_AVAILABLE' is not defined

## 5. Visualizations

### 5a. Wright Map — Persons, Items (LLTM-predicted difficulties), Rater Severities

**Interpretation**: Item difficulties are now *predicted* by the LLTM from the $Q$ matrix and $\eta$ weights. Items requiring operations with high $\eta$ (e.g., Synthesis) appear higher on the logit scale. The Wright map reveals whether the cognitive design theory embedded in $Q$ predicts a spread of difficulties that aligns with the person ability distribution. If all items cluster at one end, the test is poorly targeted for this population.

In [6]:
rater_colors = plt.cm.tab10(np.linspace(0, 0.5, R))
op_colors    = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(1, 3, width_ratios=[3, 1.5, 1.2], wspace=0.05)
ax_p = fig.add_subplot(gs[0])
ax_i = fig.add_subplot(gs[1])
ax_r = fig.add_subplot(gs[2])
y_lim = (-4, 4)

ax_p.hist(theta_est, bins=16, orientation='horizontal',
          color='steelblue', alpha=0.75, edgecolor='white')
ax_p.set_ylim(y_lim); ax_p.invert_xaxis()
ax_p.set_xlabel('Frequency', fontsize=11); ax_p.set_ylabel('Logit Scale', fontsize=11)
ax_p.set_title('Persons $\\hat{\\theta}_j$', fontsize=12)
ax_p.axhline(0, color='gray', linestyle='--', linewidth=0.8)

for i in range(I):
    b_pred = Q[i] @ eta_est
    # Color by dominant operation
    dom_op = np.argmax(Q[i] * eta_est) if Q[i].sum() > 0 else 0
    ax_i.plot([0.15, 0.85], [b_pred, b_pred],
              color=op_colors[dom_op], linewidth=1.5)
    ax_i.text(0.88, b_pred, f'{i+1}', fontsize=7, va='center')

ax_i.set_ylim(y_lim); ax_i.set_xlim(0, 1.3); ax_i.set_xticks([])
ax_i.set_yticks(range(-4, 5)); ax_i.yaxis.set_label_position('right'); ax_i.yaxis.tick_right()
ax_i.set_title('LLTM Item Difficulty $Q\\hat{\\eta}$', fontsize=11)
ax_i.axhline(0, color='gray', linestyle='--', linewidth=0.8)
for v in range(V):
    ax_i.plot([], [], color=op_colors[v], linewidth=2, label=op_names[v])
ax_i.legend(loc='lower right', fontsize=6)

for r, rv in enumerate(phi_est):
    ax_r.plot([0.1, 0.7], [rv, rv], color=rater_colors[r], linewidth=2.5)
    ax_r.text(0.75, rv, f'R{r+1}', fontsize=9, va='center', color=rater_colors[r])
ax_r.set_ylim(y_lim); ax_r.set_xlim(0, 1.2); ax_r.set_xticks([])
ax_r.set_yticks(range(-4, 5)); ax_r.yaxis.set_label_position('right'); ax_r.yaxis.tick_right()
ax_r.set_title('Rater Severity $\\hat{\\phi}_r$', fontsize=11)
ax_r.axhline(0, color='gray', linestyle='--', linewidth=0.8)

fig.suptitle('Wright Map — LLTM + Facets (Theory-Based Item Difficulty + Rater Effects)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'wright_map_lltm.png'), dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'plt' is not defined

### 5b. Item Characteristic Curves (ICC) — Grouped by Dominant Cognitive Operation

**Interpretation**: Items requiring predominantly **Synthesis** (highest $\eta$) are harder and their ICC is shifted rightward. Items requiring only **Comprehension** (lowest $\eta$) are easier and their ICC is shifted leftward. Items that share the same cognitive operation profile should have very similar ICC positions — this is the **LLTM prediction**: cognitive theory explains item difficulty. If ICCs within the same operation group overlap substantially, it validates the cognitive model.

In [7]:
theta_range  = np.linspace(-4, 4, 300)
rater_ls     = ['-', '--', '-.', ':', (0, (3,1,1,1))]

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
axes = axes.ravel()

for idx, i in enumerate(range(min(8, I))):
    ax = axes[idx]
    b_i = Q[i] @ eta_est
    for r in range(R):
        logit = theta_range - b_i - phi_est[r]
        prob  = 1.0 / (1.0 + np.exp(-logit))
        ax.plot(theta_range, prob, color=rater_colors[r],
                linestyle=rater_ls[r], linewidth=1.3, label=f'R{r+1}')
    # Show predicted difficulty
    ax.axvline(b_i, color='black', linestyle=':', linewidth=1.0)
    ops_used = [op_names[v][:4] for v in range(V) if Q[i, v]]
    ops_str = '+'.join(ops_used)
    ax.set_title(f'Item {i+1}: {ops_str}\nb={b_i:.2f}', fontsize=8)
    ax.set_xlim(-4, 4); ax.set_ylim(0, 1)
    if idx >= 4: ax.set_xlabel('$\\theta$', fontsize=9)
    if idx in [0, 4]: ax.set_ylabel('P(correct)', fontsize=9)

axes[0].legend(fontsize=7)
fig.suptitle('ICC — LLTM+Facets (items coloured by rater, dotted line = LLTM predicted difficulty)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'icc_lltm.png'), dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'np' is not defined

### 5c. Test Characteristic Curve (TCC) and Component Weight Plot

**Panel 1 — TCC**: Rater-specific TCCs, showing scoring differences due to rater severity.

**Panel 2 — Component Weight Chart**: The estimated $\hat{\eta}_v$ values with 90% credible intervals. This is the key theoretical output of LLTM — it tests whether the cognitive operations have the expected ordering of difficulty and provides estimates of their relative contributions.

**Interpretation**: If $\eta_4 (\text{Synthesis}) > \eta_3 (\text{Analysis}) > \eta_2 (\text{Application}) > \eta_1 (\text{Comprehension})$, the cognitive model is supported. Wide credible intervals on $\eta_v$ suggest insufficient statistical power (too few items per operation in the $Q$ matrix) to estimate that component reliably.

In [8]:
if not (STAN_AVAILABLE and 'fit' in dir()):
    print('ℹ️  Using true parameter values for visualization.')
    theta_est = theta_true + np.random.normal(0, 0.05, J)
    eta_est = eta_true + np.random.normal(0, 0.05, len(eta_true))
    b_est = Q @ eta_est
    phi_est = phi_true + np.random.normal(0, 0.02, R)
else:
    eta_samples = fit.stan_variable('eta')  # shape (4000, V)
    eta_ci_lo   = np.percentile(eta_samples, 5, axis=0)
    eta_ci_hi   = np.percentile(eta_samples, 95, axis=0)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # --- TCC ---
    tcc_all = []
    for r in range(R):
        tcc_r = np.zeros(len(theta_range))
        for i in range(I):
            b_i   = Q[i] @ eta_est
            logit = theta_range - b_i - phi_est[r]
            tcc_r += 1.0 / (1.0 + np.exp(-logit))
        tcc_all.append(tcc_r)
        ax1.plot(theta_range, tcc_r, color=rater_colors[r], linewidth=2,
                 label=f'Rater {r+1} ($\\phi$={phi_est[r]:.2f})')
    
    tcc_arr = np.array(tcc_all)
    ax1.fill_between(theta_range, tcc_arr.min(axis=0), tcc_arr.max(axis=0),
                     alpha=0.15, color='gray')
    ax1.set_xlabel('$\\theta$ — Person Ability', fontsize=11)
    ax1.set_ylabel('Expected Score', fontsize=11)
    ax1.set_title('TCC by Rater — LLTM+Facets', fontsize=12)
    ax1.set_xlim(-4, 4); ax1.set_ylim(0, I)
    ax1.legend(fontsize=8)
    
    # --- Component Weights ---
    x_pos = np.arange(V)
    ax2.barh(x_pos, eta_est, color=op_colors, alpha=0.75, height=0.5)
    ax2.errorbar(eta_est, x_pos,
                 xerr=[eta_est - eta_ci_lo, eta_ci_hi - eta_est],
                 fmt='none', color='black', linewidth=1.5, capsize=5)
    ax2.scatter(eta_true, x_pos, color='red', zorder=5, s=60, label='True $\\eta$')
    ax2.axvline(0, color='gray', linestyle='--', linewidth=0.8)
    ax2.set_yticks(x_pos); ax2.set_yticklabels(op_names, fontsize=10)
    ax2.set_xlabel('Component Weight ($\\hat{\\eta}_v$)', fontsize=11)
    ax2.set_title('LLTM Cognitive Component Weights\n(90% CI, red = true)', fontsize=11)
    ax2.legend(fontsize=9)
    
    plt.tight_layout()
    plt.savefig(os.path.join(tmpdir, 'tcc_lltm.png'), dpi=120, bbox_inches='tight')
    plt.show()


NameError: name 'STAN_AVAILABLE' is not defined

## 6. LLTM vs. Unrestricted Rasch Comparison

A key diagnostic is comparing LLTM-predicted item difficulties $\hat{b}_i^{LLTM} = Q_i \hat{\eta}$ against unrestricted Rasch estimates $\hat{b}_i^{Rasch}$. High correlation validates the cognitive model.

**Interpretation**: If the scatterplot shows strong alignment with the diagonal (*$R^2$ close to 1*), the cognitive operations in $Q$ account for most item difficulty variance. Systematic deviations suggest unmeasured operations or item-specific factors beyond the model.

In [9]:
if STAN_AVAILABLE:
    # Fit unrestricted Rasch model
    rasch_code = """
    data {
      int<lower=1> J; int<lower=1> I; int<lower=0> N;
      array[N] int<lower=1,upper=J> jj;
      array[N] int<lower=1,upper=I> ii;
      array[N] int<lower=0,upper=1> y;
    }
    parameters { vector[J] theta; vector[I] b; }
    model {
      theta ~ normal(0, 1); b ~ normal(0, 2);
      for (n in 1:N) y[n] ~ bernoulli_logit(theta[jj[n]] - b[ii[n]]);
    }
    """
    # Collapse rater dimension: use average response per person-item
    from collections import defaultdict
    pi_dict = defaultdict(list)
    for n in range(N):
        pi_dict[(jj_arr[n], ii_arr[n])].append(y_arr[n])
    
    jj_r, ii_r, y_r = [], [], []
    for (j, i), vals in pi_dict.items():
        jj_r.append(j); ii_r.append(i)
        y_r.append(int(round(np.mean(vals))))
    
    rasch_path = os.path.join(tmpdir, 'rasch.stan')
    with open(rasch_path, 'w') as f: f.write(rasch_code)
    rasch_model = cmdstanpy.CmdStanModel(stan_file=rasch_path)
    rasch_fit   = rasch_model.sample(
        data={'J': J, 'I': I, 'N': len(jj_r), 'jj': jj_r, 'ii': ii_r, 'y': y_r},
        chains=2, iter_warmup=500, iter_sampling=500, seed=42, show_progress=False
    )
    b_rasch = rasch_fit.stan_variable('b').mean(axis=0)
    
    lltm_b_pred = Q @ eta_est
    r2 = np.corrcoef(b_rasch, lltm_b_pred)[0,1]**2
    
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(b_rasch, lltm_b_pred, c=[op_colors[np.argmax(Q[i] * eta_est)] for i in range(I)],
               s=80, zorder=3, edgecolors='white')
    for i in range(I):
        ax.annotate(f'{i+1}', (b_rasch[i], lltm_b_pred[i]),
                    fontsize=7, ha='left', va='bottom')
    lims = [min(b_rasch.min(), lltm_b_pred.min()) - 0.2,
            max(b_rasch.max(), lltm_b_pred.max()) + 0.2]
    ax.plot(lims, lims, 'k--', linewidth=1.2, label='Perfect fit')
    ax.set_xlabel('Unrestricted Rasch $\\hat{b}_i$', fontsize=12)
    ax.set_ylabel('LLTM Predicted $Q_i \\hat{\\eta}$', fontsize=12)
    ax.set_title(f'LLTM Validity Check: Predicted vs. Rasch Difficulty\n$R^2 = {r2:.3f}$',
                 fontsize=12)
    ax.set_xlim(lims); ax.set_ylim(lims)
    for v in range(V):
        ax.scatter([], [], c=op_colors[v], s=60, label=op_names[v])
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(tmpdir, 'lltm_validity.png'), dpi=120, bbox_inches='tight')
    plt.show()
else:
    print("⚠️  Stan not available — skipping model compilation/fitting.")


NameError: name 'STAN_AVAILABLE' is not defined

In [ ]:
# ── Posterior Parameter Density (Logit Scale) ─────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
fig.suptitle('LLTM+Facets — Estimated Parameter Distributions (Logit Scale)',
             fontsize=13, fontweight='bold')

panels = [
    (axes[0], theta_est, r'$\hat{\theta}_j$  (person ability)',     'steelblue'),
    (axes[1], b_est,     r'$\hat{b}_i = Q\hat{\eta}$  (item diff)', 'firebrick'),
    (axes[2], eta_est,   r'$\hat{\eta}_p$  (operation weights)',     'darkorange'),
    (axes[3], phi_est,   r'$\hat{\phi}_r$  (rater severity)',        'mediumpurple'),
]

for ax, vals, title, color in panels:
    ax.hist(vals, bins=max(8, len(vals)//2), density=True,
            color=color, alpha=0.35, edgecolor='white')
    if len(vals) >= 3:
        xs = np.linspace(vals.min() - 0.5, vals.max() + 0.5, 300)
        kde = gaussian_kde(vals, bw_method='scott')
        ax.plot(xs, kde(xs), color=color, linewidth=2)
    ax.axvline(vals.mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'mean={vals.mean():.2f}')
    ax.axvline(0, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.set_xlabel('Logit', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(f'{title}  (n={len(vals)})', fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'density_lltm.png'), dpi=120, bbox_inches='tight')
plt.show()
for name, vals in [('theta', theta_est), ('b (LLTM)', b_est), ('eta', eta_est), ('phi', phi_est)]:
    print(f"{name:12s}: mean={vals.mean():.3f}  SD={vals.std():.3f}  "
          f"range=[{vals.min():.2f}, {vals.max():.2f}]")
